# Gradient Descent Visualization
This notebook allows you to input mathematical functions (1D or 2D), automatically computes their gradients, and visualizes the optimization process using Gradient Descent.

In [53]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, clear_output
from mpl_toolkits.mplot3d import Axes3D

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

In [54]:
def robust_lambdify(args, expr):
    """
    Wraps sympy.lambdify to ensure it always returns an array 
    if the input contains arrays, even for constant functions.
    """
    f = sp.lambdify(args, expr, modules='numpy')
    
    def wrapper(*input_args):
        result = f(*input_args)
        # Check if result is scalar but inputs were arrays
        if np.isscalar(result):
            if any(isinstance(arg, np.ndarray) for arg in input_args):
                shape = np.broadcast(*input_args).shape
                return np.full(shape, result)
        return result
    
    return wrapper

def get_func_and_grads(expr_str):
    """
    Parses a string expression into callable numpy functions for f, df/dx, and df/dy.
    """
    x, y = sp.symbols('x y')
    try:
        expr = sp.sympify(expr_str)
    except Exception as e:
        return None, None, None, f"Error parsing expression: {e}"

    # Determine variables present
    free_symbols = expr.free_symbols
    
    # Validation: Only allow x and y
    for sym in free_symbols:
        if str(sym) not in ['x', 'y']:
            return None, None, None, f"Error: Unknown variable '{sym}'. Only 'x' and 'y' are allowed."

    # Calculate gradients
    grad_x = sp.diff(expr, x)
    grad_y = sp.diff(expr, y)

    # Lambdify
    # Note: We always lambdify with (x, y) to keep signature consistent, 
    # even if one variable is missing.
    f = robust_lambdify((x, y), expr)
    df_dx = robust_lambdify((x, y), grad_x)
    df_dy = robust_lambdify((x, y), grad_y)
    
    is_2d = 'y' in [str(s) for s in free_symbols]
    
    return (f, df_dx, df_dy), is_2d, None

In [55]:
def run_optimizer(start_x, start_y, df_dx, df_dy, learning_rate, max_iter, noise_scale=0.0, momentum=0.0, optimizer='GD'):
    """
    Performs optimization using GD (with momentum), Adagrad, or Adam.
    """
    path = []
    curr_x, curr_y = start_x, start_y
    path.append((curr_x, curr_y))
    
    # State variables
    # GD: vx is velocity
    # Adam: vx is m (1st moment), sx is v (2nd moment)
    # Adagrad: sx is sum of squares
    vx, vy = 0.0, 0.0 
    sx, sy = 0.0, 0.0 
    
    # Adam hyperparameters
    beta1 = 0.9
    beta2 = 0.999
    epsilon = 1e-8
    
    for t in range(1, max_iter + 1):
        grad_x = df_dx(curr_x, curr_y)
        grad_y = df_dy(curr_x, curr_y)
        
        # Add noise to simulate SGD
        if noise_scale > 0:
            grad_x += np.random.normal(0, noise_scale)
            grad_y += np.random.normal(0, noise_scale)
            
        if optimizer == 'GD':
            # Momentum update
            vx = momentum * vx + grad_x
            vy = momentum * vy + grad_y
            
            curr_x -= learning_rate * vx
            curr_y -= learning_rate * vy
            
        elif optimizer == 'Adagrad':
            sx += grad_x**2
            sy += grad_y**2
            
            curr_x -= learning_rate * grad_x / (np.sqrt(sx) + epsilon)
            curr_y -= learning_rate * grad_y / (np.sqrt(sy) + epsilon)
            
        elif optimizer == 'Adam':
            vx = beta1 * vx + (1 - beta1) * grad_x
            vy = beta1 * vy + (1 - beta1) * grad_y
            
            sx = beta2 * sx + (1 - beta2) * grad_x**2
            sy = beta2 * sy + (1 - beta2) * grad_y**2
            
            # Bias correction
            vx_hat = vx / (1 - beta1**t)
            vy_hat = vy / (1 - beta1**t)
            sx_hat = sx / (1 - beta2**t)
            sy_hat = sy / (1 - beta2**t)
            
            curr_x -= learning_rate * vx_hat / (np.sqrt(sx_hat) + epsilon)
            curr_y -= learning_rate * vy_hat / (np.sqrt(sy_hat) + epsilon)
            
        path.append((curr_x, curr_y))
        
        # Convergence check (only for deterministic GD)
        if noise_scale == 0 and optimizer == 'GD' and momentum == 0 and np.sqrt(grad_x**2 + grad_y**2) < 1e-6:
            break
            
    return np.array(path)

In [56]:
def plot_1d(f, path, x_range=(-10, 10)):
    x = np.linspace(x_range[0], x_range[1], 400)
    # Pass y=0 for 1D functions since our signature expects (x,y)
    y_vals = f(x, np.zeros_like(x))
    
    plt.figure(figsize=(10, 6))
    plt.plot(x, y_vals, label='f(x)')
    
    # Plot path
    path_x = path[:, 0]
    path_y = f(path_x, np.zeros_like(path_x))
    
    plt.scatter(path_x, path_y, c='red', s=50, zorder=5, label='Steps')
    plt.plot(path_x, path_y, 'r--', alpha=0.5)
    
    # Plot gradient arrows (tangent lines)
    # We can approximate the direction by looking at the next step
    for i in range(len(path_x) - 1):
        dx = path_x[i+1] - path_x[i]
        dy = path_y[i+1] - path_y[i]
        # Scale arrow for visibility
        plt.arrow(path_x[i], path_y[i], dx, dy, 
                  head_width=0.2, head_length=0.3, fc='green', ec='green', alpha=0.6)

    plt.title("Gradient Descent (1D)")
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.legend()
    plt.show()

def plot_2d(f, path, df_dx, df_dy, x_range=(-5, 5), y_range=(-5, 5)):
    x = np.linspace(x_range[0], x_range[1], 30) # Reduced resolution for quiver
    y = np.linspace(y_range[0], y_range[1], 30)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)
    
    # Calculate gradients for quiver plot
    U = -df_dx(X, Y) # Negative gradient points to minima
    V = -df_dy(X, Y)
    
    # High res for contour
    x_hi = np.linspace(x_range[0], x_range[1], 100)
    y_hi = np.linspace(y_range[0], y_range[1], 100)
    X_hi, Y_hi = np.meshgrid(x_hi, y_hi)
    Z_hi = f(X_hi, Y_hi)

    fig = plt.figure(figsize=(16, 7))
    
    # 1. Contour Plot with Quiver
    ax1 = fig.add_subplot(1, 2, 1)
    contour = ax1.contourf(X_hi, Y_hi, Z_hi, levels=20, cmap='viridis')
    plt.colorbar(contour, ax=ax1)
    
    # Quiver plot (Gradient Field)
    # Normalize arrows for better visualization
    norm = np.sqrt(U**2 + V**2)
    # Avoid division by zero
    norm[norm == 0] = 1
    U, V = U/norm, V/norm
    
    ax1.quiver(X, Y, U, V, color='white', alpha=0.3, headwidth=3, headlength=4)
    
    # Plot path
    path_x = path[:, 0]
    path_y = path[:, 1]
    ax1.plot(path_x, path_y, 'r.-', markersize=10, label='Path')
    ax1.scatter(path_x[0], path_y[0], c='white', s=100, marker='*', label='Start')
    ax1.scatter(path_x[-1], path_y[-1], c='red', s=100, marker='x', label='End')
    
    ax1.set_title("Contour Plot with Gradient Field")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.legend()
    
    # 2. 3D Surface Plot
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    surf = ax2.plot_surface(X_hi, Y_hi, Z_hi, cmap='viridis', alpha=0.8, edgecolor='none')
    
    # Plot path on 3D surface
    path_z = f(path_x, path_y)
    ax2.plot(path_x, path_y, path_z, 'r.-', linewidth=2, markersize=5, zorder=10)
    
    ax2.set_title("3D Surface Plot")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y")
    ax2.set_zlabel("f(x, y)")
    
    plt.tight_layout()
    plt.show()

In [57]:
def run_app(expr, lr, iterations, start_x, start_y, noise_scale, momentum, optimizer):
    funcs, is_2d, error = get_func_and_grads(expr)
    
    if error:
        print(error)
        return
        
    f, df_dx, df_dy = funcs
    
    # Run Optimization
    path = run_optimizer(start_x, start_y, df_dx, df_dy, lr, iterations, noise_scale, momentum, optimizer)
    
    print(f"Optimizer: {optimizer}")
    print(f"Final Position: ({path[-1,0]:.4f}, {path[-1,1]:.4f})")
    print(f"Final Value: {f(path[-1,0], path[-1,1]):.4f}")
    
    if is_2d:
        # Adjust plot range dynamically based on path
        margin = 2
        x_min, x_max = min(path[:,0].min(), -5), max(path[:,0].max(), 5)
        y_min, y_max = min(path[:,1].min(), -5), max(path[:,1].max(), 5)
        # Pass gradient functions to plot_2d
        plot_2d(f, path, df_dx, df_dy, x_range=(x_min-margin, x_max+margin), y_range=(y_min-margin, y_max+margin))
    else:
        margin = 2
        x_min, x_max = min(path[:,0].min(), -5), max(path[:,0].max(), 5)
        plot_1d(f, path, x_range=(x_min-margin, x_max+margin))

# Create Widgets
style = {'description_width': 'initial'}

expr_widget = widgets.Text(value='x**2 + y**2', description='Function f(x,y):', style=style)
lr_widget = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.1, description='Learning Rate:', style=style)
iter_widget = widgets.IntSlider(value=20, min=1, max=200, step=1, description='Iterations:', style=style)
start_x_widget = widgets.FloatSlider(value=4.0, min=-10, max=10, step=0.1, description='Start X:', style=style)
start_y_widget = widgets.FloatSlider(value=3.0, min=-10, max=10, step=0.1, description='Start Y:', style=style)
noise_widget = widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description='SGD Noise:', style=style)
momentum_widget = widgets.FloatSlider(value=0.0, min=0.0, max=0.99, step=0.01, description='Momentum:', style=style)
optim_widget = widgets.Dropdown(options=['GD', 'Adagrad', 'Adam'], value='GD', description='Optimizer:', style=style)

ui = widgets.VBox([
    expr_widget,
    widgets.HBox([optim_widget, lr_widget, iter_widget]),
    widgets.HBox([noise_widget, momentum_widget]),
    widgets.HBox([start_x_widget, start_y_widget])
])

out = widgets.interactive_output(run_app, {
    'expr': expr_widget,
    'lr': lr_widget,
    'iterations': iter_widget,
    'start_x': start_x_widget,
    'start_y': start_y_widget,
    'noise_scale': noise_widget,
    'momentum': momentum_widget,
    'optimizer': optim_widget
})

display(ui, out)

Output()

In [60]:
# ------------------------------------------------------------------------------
# MANUAL INPUT MODE
# ------------------------------------------------------------------------------
# Run this cell to input a function via a text prompt.
# Then interact with the parameters using sliders.

func_input = input("Enter function expression (e.g., x**2 + y**2): ")

if func_input:
    print(f"Interactive Mode for: {func_input}")
    
    # Create widgets for parameters
    style = {'description_width': 'initial'}
    lr_widget_man = widgets.FloatLogSlider(value=0.1, base=10, min=-3, max=0, step=0.1, description='Learning Rate:', style=style)
    iter_widget_man = widgets.IntSlider(value=30, min=1, max=200, step=1, description='Iterations:', style=style)
    start_x_widget_man = widgets.FloatSlider(value=4.0, min=-10, max=10, step=0.1, description='Start X:', style=style)
    start_y_widget_man = widgets.FloatSlider(value=4.0, min=-10, max=10, step=0.1, description='Start Y:', style=style)
    noise_widget_man = widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description='SGD Noise:', style=style)
    momentum_widget_man = widgets.FloatSlider(value=0.0, min=0.0, max=0.99, step=0.01, description='Momentum:', style=style)
    optim_widget_man = widgets.Dropdown(options=['GD', 'Adagrad', 'Adam'], value='GD', description='Optimizer:', style=style)

    # UI Layout
    ui_man = widgets.VBox([
        widgets.HBox([optim_widget_man, lr_widget_man, iter_widget_man]),
        widgets.HBox([noise_widget_man, momentum_widget_man]),
        widgets.HBox([start_x_widget_man, start_y_widget_man])
    ])

    # Wrapper to pass the fixed function string
    def run_manual_interaction(lr, iterations, start_x, start_y, noise_scale, momentum, optimizer):
        run_app(func_input, lr, iterations, start_x, start_y, noise_scale, momentum, optimizer)

    out_man = widgets.interactive_output(run_manual_interaction, {
        'lr': lr_widget_man,
        'iterations': iter_widget_man,
        'start_x': start_x_widget_man,
        'start_y': start_y_widget_man,
        'noise_scale': noise_widget_man,
        'momentum': momentum_widget_man,
        'optimizer': optim_widget_man
    })

    display(ui_man, out_man)

Interactive Mode for: 1/20*x*x+y*y


Output()

---

## 🚀 Deployment to GitHub Pages

This notebook can be deployed to GitHub Pages using **JupyterLite**, allowing users to interact with the code and widgets directly in their browser.

### Prerequisites
- Python environment with JupyterLite installed
- Git and GitHub repository

### Steps to Deploy

#### 1. Install JupyterLite and Dependencies
```bash
.\simulations\Scripts\python -m pip install jupyterlite-core jupyterlite-pyodide-kernel jupyter-server
```

#### 2. Build the JupyterLite Site
```bash
.\simulations\Scripts\jupyter lite build --contents . --output-dir docs
```

**Note:** If you get an error about missing packages during build, also install:
```bash
.\simulations\Scripts\python -m pip install notebook
```

#### 3. Deploy to GitHub
1. Commit and push the `docs` folder to your repository
2. In GitHub: Settings → Pages → Source: Deploy from a branch
3. Select branch: **main**, folder: **/docs**
4. Save and wait for deployment

#### 4. Access Your Live Notebook
Your notebook will be available at:
```
https://[username].github.io/[repo]/lab/index.html
```

### Features Available in Deployed Version
- ✅ Full Jupyter Lab interface in browser
- ✅ All code cells visible and editable
- ✅ Interactive widgets work seamlessly
- ✅ Users can modify and re-run code
- ✅ No server required (runs via WebAssembly)
- ✅ Supports numpy, matplotlib, sympy out of the box

### Example Functions to Try
- **Simple Bowl**: `x**2 + y**2`
- **Rosenbrock**: `(1-x)**2 + 100*(y-x**2)**2`
- **Himmelblau**: `(x**2 + y - 11)**2 + (x + y**2 - 7)**2`
- **Beale**: `(1.5 - x + x*y)**2 + (2.25 - x + x*y**2)**2`
- **1D Parabola**: `x**2 - 4*x + 5`

### Optimizer Tips
- **Adam vs GD**: Try `x**2 + 10*y**2` to see adaptive learning rates
- **Momentum**: Set to 0.9 on `x**2 + y**2` to see acceleration
- **SGD Noise**: Add noise ~1.0 to observe stochastic behavior
- **Adagrad**: Great for functions with varying gradient magnitudes